In [ ]:
import tensorflow as tf
import numpy as np
import math
import matplotlib.pyplot as plt
import os

In [ ]:
def build_generator(noise_inputs,label_inputs,image_size = 28):
    x = tf.keras.layers.concatenate([noise_inputs,label_inputs],axis = 1)
    
    x = tf.keras.layers.Dense(7 * 7 * 128)(input_layer)
    x = tf.keras.layers.Reshape((7,7,128))(x)
    
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2DTranspose(128,kernel_size = [5,5],strides = 2,padding = 'same')(x)
    
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2DTranspose(64,kernel_size = [5,5],strides = 2,padding = 'same')(x)
    
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2DTranspose(32,kernel_size = [5,5],strides = 1,padding = 'same')(x)
    
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2DTranspose(1,kernel_size = [5,5],strides = 1,padding = 'same')(x)
    
    x = tf.keras.layers.Activation('sigmoid')(x)
    gen_network = tf.keras.models.Model(input_layer,x,name = 'gen_network')

    return gen_network

In [ ]:
def build_discriminator(image_inputs,label_inputs,image_size = 28):
    filter_size = 5
    num_filters = [32,64,128,256]
    stride_size = [2,2,2,1]
    
    x = image_inputs
    
    y = tf.keras.layers.Dense(28*28)(label_inputs)
    y = tf.keras.layers.Reshape((28,28,1))(y)
    x = tf.keras.layers.concatenate([x,y])
    
    x = tf.keras.layers.LeakyReLU(alpha = 0.2)(x)
    x = tf.keras.layers.Conv2D(32,kernel_size = [5,5],strides = 2,padding = 'same')(x)
    
    x = tf.keras.layers.LeakyReLU(alpha = 0.2)(disc_input)
    x = tf.keras.layers.Conv2D(64,kernel_size = [5,5],strides = 2,padding = 'same')(x)
    
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(1,activation = 'sigmoid')(x)
    
    disc_network = tf.keras.models.Model(x,y,name = 'disc_network')
    
    return disc_network

In [ ]:
import tensorflow as tf
from tensorflow.keras.optimizers.schedules import ExponentialDecay

def build_models():
    noise_size = 100
    initial_lr = 2e-4

    # Define learning rate schedules
    lr_schedule_discriminator = ExponentialDecay(
        initial_learning_rate=initial_lr,
        decay_steps=10000,
        decay_rate=0.96,
        staircase=True
    )

    lr_schedule_adversarial = ExponentialDecay(
        initial_learning_rate=initial_lr * 0.5,
        decay_steps=10000,
        decay_rate=0.96,
        staircase=True
    )

    noise_inputs = tf.keras.layers.Input(shape = (noise_size))
    label_inputs = tf.keras.layers.Input(shape = (10,))
    noise_inputs = tf.keras.layers.Input(shape = (28,28,1,))
    
    # Build Base Discriminator model
    base_discriminator = build_discriminator(image_inputs,label_inputs)

    # Define optimizer and compile discriminator
    discriminator = tf.keras.models.Model(inputs=base_discriminator.inputs,
                                          outputs=base_discriminator.outputs)
    optimizer_d = tf.keras.optimizers.RMSprop(learning_rate=lr_schedule_discriminator)
    discriminator.compile(loss='binary_crossentropy',
                          optimizer=optimizer_d,
                          metrics=['accuracy'])

    # Build Generator model
    generator = build_generator(image_size=28, noise_input=noise_size)

    # Build Frozen Discriminator for adversarial training
    frozen_discriminator = tf.keras.models.Model(inputs=base_discriminator.inputs,
                                                 outputs=base_discriminator.outputs)
    frozen_discriminator.trainable = False

    # Build Adversarial model
    optimizer_a = tf.keras.optimizers.RMSprop(learning_rate=lr_schedule_adversarial)
    adversarial = tf.keras.models.Model(generator.input,
                                        frozen_discriminator(generator.output))
    adversarial.compile(loss='binary_crossentropy',
                        optimizer=optimizer_a,
                        metrics=['accuracy'])

    return generator, discriminator, adversarial